# Exploration du jeu de données sur les mesures de radiations prélevées

Ce notebook sert d'exploration au jeu de données d'openradiation pour pouvoir comprendre la structure des trois fichiers du dossier tar mise à jour une fois par jour.

On commencera dans un premier temps par une ingestion de ce fichier et ensuite extraire les trois fichiers CSV conservées et les explorer.

Cela nous permettra de déterminer le chemin à suivre pour l'ingestion, le traitement et l'analyse de ces données.

In [14]:
# Chargement des librairies
import requests
from pyspark.sql import SparkSession
import tarfile
from notebookutils import mssparkutils # La gestion du lakehouse
from pathlib import Path
import os

StatementMeta(, 56081b58-39e9-42f3-b70c-38f20de855ab, 16, Finished, Available, Finished, False)

# Capturer le jeu de données via l'url suivante

https://request.openradiation.net/openradiation_dataset.tar.gz

In [2]:
# Définition de l'url de requête à appliquer
URL = (
    "https://request.openradiation.net/"
    "openradiation_dataset.tar.gz"
)


StatementMeta(, 56081b58-39e9-42f3-b70c-38f20de855ab, 4, Finished, Available, Finished, False)

In [7]:
# Requête Get et récupération de la réponse
response = requests.get(
        URL,
        headers={"User-Agent": "Python OpenRadiation downloader"},
    )

print(f"Requête GET URL fichier tar à récupérer : {URL}")
status = response.status_code

if status != 200:
            raise RuntimeError(f"HTTP inattendu : {status}")

StatementMeta(, 56081b58-39e9-42f3-b70c-38f20de855ab, 9, Finished, Available, Finished, False)

Requête GET URL fichier tar à récupérer : https://request.openradiation.net/openradiation_dataset.tar.gz


Enregistrons le fichier résultant dans notre lakehouse (openradiation_lakehouse)

In [10]:
LAKEHOUSE_PATH = "Files/OpenRadiation" # Destination de mon fichier

StatementMeta(, 56081b58-39e9-42f3-b70c-38f20de855ab, 12, Finished, Available, Finished, False)

In [11]:
# Créer le dossier de destination
mssparkutils.fs.mkdirs(LAKEHOUSE_PATH)

# Écrire le contenu binaire dans un fichier temporaire local
local_path = "/tmp/openradiation_dataset.tar.gz"

with open(local_path, "wb") as f:
    f.write(response.content)

# Copier le fichier dans le Lakehouse
mssparkutils.fs.cp(
    f"file:{local_path}",
    LAKEHOUSE_PATH+"/openradiation_dataset.tar.gz"
)

print(f"Archive enregistrée dans le Lakehouse : {LAKEHOUSE_PATH}")
print(f"Taille : {len(response.content):,} octets")

StatementMeta(, 56081b58-39e9-42f3-b70c-38f20de855ab, 13, Finished, Available, Finished, False)

Archive enregistrée dans le Lakehouse : Files/OpenRadiation
Taille : 441,479,313 octets


Le fichier a bien été enregistré.

# Extraction des fichiers dans le fichier compressé 

Maintenant que nous avons collecté le fichier compressé, extrayons ces différents fichiers.

Définissons d'abord les chemins temporaires et de destinations.

In [15]:
# Répertoire temporaire local pour les fichiers CSV
TEMP_DIR = "/tmp/openradiation_extract"

# Destination finale dans le Lakehouse
LAKEHOUSE_DIR = "Files/OpenRadiation/extracted"

os.makedirs(TEMP_DIR, exist_ok=True)
mssparkutils.fs.mkdirs(LAKEHOUSE_DIR)

# Extraction des fichiers CSV uniquement
csv_files = []

StatementMeta(, 56081b58-39e9-42f3-b70c-38f20de855ab, 17, Finished, Available, Finished, False)

Lisons le fichier compressé et extraire les fichiers CSV qui s'y trouve.

In [24]:
# Chemin pour lire sera celui-ci
print(os.listdir("/lakehouse/default/Files/OpenRadiation/"))
ARCHIVE_PATH="/lakehouse/default/Files/OpenRadiation/openradiation_dataset.tar.gz"

StatementMeta(, 56081b58-39e9-42f3-b70c-38f20de855ab, 26, Finished, Available, Finished, False)

['extracted', 'openradiation_dataset.tar.gz']


In [25]:
# Ouverture du fichier compressé et récupération des fichiers CSV
with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
    for member in tar.getmembers():
        if not member.isfile():
            continue

        # Ne retenir que les CSV
        if not member.name.lower().endswith(".csv"):
            continue

        # Éviter les chemins dangereux dans l'archive
        filename = Path(member.name).name
        if not filename:
            continue

        source = tar.extractfile(member)
        if source is None:
            continue

        local_file = Path(TEMP_DIR) / filename

        with source, local_file.open("wb") as output:
            output.write(source.read())

        csv_files.append(local_file)

# Copier les CSV vers le Lakehouse
for csv_file in csv_files:
    destination = f"{LAKEHOUSE_DIR}/{csv_file.name}"

    mssparkutils.fs.cp(
        f"file:{csv_file}",
        destination
    )

    print(f"Fichier extrait : {destination}")

print(f"\nNombre de fichiers CSV extraits : {len(csv_files)}")

StatementMeta(, 56081b58-39e9-42f3-b70c-38f20de855ab, 27, Finished, Available, Finished, False)

Fichier extrait : Files/OpenRadiation/extracted/measurements.csv
Fichier extrait : Files/OpenRadiation/extracted/tags.csv
Fichier extrait : Files/OpenRadiation/extracted/measurements_withoutEnclosedObject.csv

Nombre de fichiers CSV extraits : 3


Maintenant que nous avons extraits les fichiers CSV, lisons les avec Spark et observons leur stucture.

# Observation des différents jeu de données

Les fichiers CSV sont maintenant extraites, il ne reste plus qu'à les lire et à comprendre leur stucture.

In [34]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

def analyser_dataframe(df: DataFrame) -> None:
    """
    Analyse un DataFrame Spark :
    - nombre de lignes et de colonnes
    - proportion de données manquantes par colonne
    - structure et types des colonnes
    """

    # 1. Nombre de lignes et de colonnes
    nb_lignes = df.count()
    nb_colonnes = len(df.columns)

    print("Nombre de lignes / colonnes", nb_lignes, "/", nb_colonnes)

    # 2. Structure des colonnes
    print("\n--- Structure des colonnes ---")
    df.printSchema()

    # 3. Données manquantes par colonne
    print("\n--- Données manquantes par colonne ---")

    if nb_lignes == 0:
        print("Le DataFrame est vide : proportions non calculables.")
        return

    expressions = []

    for colonne, type_colonne in df.dtypes:
        # NULL pour tous les types ; chaîne vide ou espaces pour les chaînes
        condition_manquante = F.col(colonne).isNull()

        if type_colonne == "string":
            condition_manquante = (
                condition_manquante
                | (F.trim(F.col(colonne)) == "")
            )

        expressions.extend([
            F.sum(
                F.when(condition_manquante, 1).otherwise(0)
            ).alias(f"{colonne}__manquants")
        ])

    resultats = df.agg(*expressions).first()

    for colonne, type_colonne in df.dtypes:
        nb_manquants = resultats[f"{colonne}__manquants"]
        proportion = nb_manquants / nb_lignes * 100

        print(
            f"Colonne : {colonne} | "
            f"Type : {type_colonne} | "
            f"Manquants : {nb_manquants} | "
            f"Proportion : {proportion:.2f} %"
        )



StatementMeta(, 56081b58-39e9-42f3-b70c-38f20de855ab, 36, Finished, Available, Finished, False)

In [28]:
def read_csv_file(spark, path):
    """
    Permets de lire un fichier CSV avec ";" comme séparation 
    et de le renvoyer en Spark DataFrame
    """
    df = (
            spark.read
            .option("header", True)
            .option("sep", ";")
            .option("inferSchema", True)
            .csv(path)
        )
    return df


StatementMeta(, 56081b58-39e9-42f3-b70c-38f20de855ab, 30, Finished, Available, Finished, False)

In [29]:
# Lecture des différents fichiers CSV
files = mssparkutils.fs.ls(LAKEHOUSE_DIR)

for file in files:
    if file.name.lower().endswith(".csv"):
        print(f"\nLecture de {file.name}")

        df = read_csv_file(spark, file.path)

        df.printSchema()
        df.show(5, truncate=False)

StatementMeta(, 56081b58-39e9-42f3-b70c-38f20de855ab, 31, Finished, Available, Finished, False)


Lecture de measurements.csv
root
 |-- apparatusId: string (nullable = true)
 |-- apparatusVersion: string (nullable = true)
 |-- apparatusSensorType: string (nullable = true)
 |-- apparatusTubeType: string (nullable = true)
 |-- temperature: string (nullable = true)
 |-- value: double (nullable = true)
 |-- hitsNumber: string (nullable = true)
 |-- calibrationFunction: string (nullable = true)
 |-- startTime: string (nullable = true)
 |-- endTime: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- accuracy: double (nullable = true)
 |-- altitude: integer (nullable = true)
 |-- altitudeAccuracy: double (nullable = true)
 |-- endLatitude: double (nullable = true)
 |-- endLongitude: double (nullable = true)
 |-- endAccuracy: string (nullable = true)
 |-- endAltitude: string (nullable = true)
 |-- endAltitudeAccuracy: string (nullable = true)
 |-- deviceUuid: string (nullable = true)
 |-- devicePlatform: string (nullable = true)


Associons une variable chacune pour les observer séparément

In [30]:
measurements = read_csv_file(spark, LAKEHOUSE_DIR+"/measurements.csv")
measurements_without_enclobject = read_csv_file(spark, LAKEHOUSE_DIR+"/measurements_withoutEnclosedObject.csv")
tags = read_csv_file(spark, LAKEHOUSE_DIR+"/tags.csv")

StatementMeta(, 56081b58-39e9-42f3-b70c-38f20de855ab, 32, Finished, Available, Finished, False)

## Observation de la structure pour measurements.csv

In [32]:
print("Nombre de ligne et de colonne : ", measurements.count(), " / ", len(measurements.columns))
measurements.printSchema()

StatementMeta(, 56081b58-39e9-42f3-b70c-38f20de855ab, 34, Finished, Available, Finished, False)

Nombre de ligne et de colonne :  1992982  /  60
root
 |-- apparatusId: string (nullable = true)
 |-- apparatusVersion: string (nullable = true)
 |-- apparatusSensorType: string (nullable = true)
 |-- apparatusTubeType: string (nullable = true)
 |-- temperature: string (nullable = true)
 |-- value: double (nullable = true)
 |-- hitsNumber: string (nullable = true)
 |-- calibrationFunction: string (nullable = true)
 |-- startTime: string (nullable = true)
 |-- endTime: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- accuracy: double (nullable = true)
 |-- altitude: integer (nullable = true)
 |-- altitudeAccuracy: double (nullable = true)
 |-- endLatitude: double (nullable = true)
 |-- endLongitude: double (nullable = true)
 |-- endAccuracy: string (nullable = true)
 |-- endAltitude: string (nullable = true)
 |-- endAltitudeAccuracy: string (nullable = true)
 |-- deviceUuid: string (nullable = true)
 |-- devicePlatform: string

### Description du jeu de données 


<table>
<tr><th>API Name</th><th>Type</th><th>Available for request API</th><th>Available for submit API</th><th>Description</th></tr>
<tr><td>apparatusId</td><td>String</td><td></td><td></td><td>Unique sensor identifier</td></tr>
<tr><td>apparatusVersion</td><td>String</td><td> </td><td> </td><td>Sensor firmware version</td></tr>
<tr><td>apparatusSensorType</td><td>String</td><td> </td><td> </td><td>Sensor type : geiger, photodiode</td></tr>
<tr><td>apparatusTubeType</td><td>String</td><td> </td><td> </td><td>Tube identification (only if apparatusSensorType = geiger)</td></tr>
<tr><td>temperature</td><td>Integer</td><td></td><td></td><td>Temperature (°C) </td></tr>
<tr><td>value</td><td>Real</td><td>*</td><td>Mandatory</td><td>Dose rate (µSv/h)</td></tr>
<tr><td>hitsNumber</td><td>Integer</td><td></td><td></td><td>Counts measured during the whole measurement period. cps (counts per second) used to evaluate dose rate with the calibration function will be : cps = hitsNumber / (endTime - startTime)</td></tr>
<tr><td>calibrationFunction</td><td>String</td><td></td><td></td><td>Calibration function used to calculate µSv/h from cps (counts per second). Format should be inline with these symbols : cps 0-9. -+/*^() max,. Example : 7.543*(cps-0.02)^2+0.001*(cps-0.02)</td></tr>
<tr><td>startTime</td><td>Timestamp</td><td>*</td><td>Mandatory</td><td>Date of the beginning of the measurement (ISO GMT)</td></tr>
<tr><td>endTime</td><td>Timestamp</td><td></td><td> </td><td>Date of the end of the measurement (ISO GMT)</td></tr>
<tr><td>latitude</td><td>Real</td><td>*</td><td>Mandatory</td><td>latitude</td></tr>
<tr><td>longitude</td><td>Real</td><td>*</td><td>Mandatory</td><td>longitude</td></tr>
<tr><td>accuracy</td><td>Real</td><td></td><td> </td><td>Position accuracy in meters</td></tr>
<tr><td>altitude</td><td>Integer</td><td></td><td> </td><td>Altitude above sea in meters</td></tr>
<tr><td>altitudeAccuracy</td><td>Real</td><td></td><td> </td><td>Altitude accuracy in meters</td></tr>
<tr><td>endLatitude</td><td>Real</td><td></td><td></td><td>latitude at the end of the measurement</td></tr>
<tr><td>endLongitude</td><td>Real</td><td></td><td></td><td>longitude at the end of the measurement</td></tr>
<tr><td>endAccuracy</td><td>Real</td><td></td><td></td><td>Position accuracy in meters at the end of the measurement</td></tr>
<tr><td>endAltitude</td><td>Integer</td><td></td><td></td><td>Altitude above sea in meters at the end of the measurement</td></tr>
<tr><td>endAltitudeAccuracy</td><td>Real</td><td></td><td></td><td>Altitude accuracy in meters at the end of the measurement</td></tr>
<tr><td>deviceUuid</td><td>String</td><td></td><td> </td><td>Smartphone device UUID  (see http://plugins.cordova.io/#/package/org.apache.cordova.device)</td></tr>
<tr><td>devicePlatform</td><td>String</td><td></td><td> </td><td>Smartphone device platform</td></tr>
<tr><td>deviceVersion</td><td>String</td><td></td><td> </td><td>Smartphone device OS version</td></tr>
<tr><td>deviceModel</td><td>String</td><td></td><td> </td><td>Smartphone device model</td></tr>
<tr><td>reportUuid</td><td>String</td><td>*</td><td>Mandatory</td><td>Unique measurement UUID (UUIDv4 format) client-side generated</td></tr>
<tr><td>manualReporting</td><td>Boolean</td><td></td><td></td><td>Manual Reporting : true, false (default:true). False if the data is not entered by a human being</td></tr>
<tr><td>organisationReporting</td><td>String</td><td></td><td></td><td>Software version (sample:openradiation_v1)</td></tr>
<tr><td>reportContext</td><td>String</td><td>Never</td><td></td><td>Report context : emergency, routine, exercise, test (default:test). test:data are not registrated but you can test api use, emergency and exercise:not used,routine:you should use this one !</td></tr>
<tr><td>description</td><td>String</td><td></td><td></td><td>Free description (only if userId is specified)</td></tr>
<tr><td>measurementHeight</td><td>Integer</td><td></td><td></td><td>Measurement height above the ground (in meters). It is measurement height from the ground (and not altitude above sea level).</td></tr>
<tr><td>tags</td><td>Json array of string</td><td></td><td></td><td>Free tags list [tag1 ; tag2] (only if userId is specified)</td></tr>
<tr><td>enclosedObject</td><td>String</td><td></td><td></td><td>Base64 encoded Image. The size shoudn't exceeded 1mb and format should be closed from 600*800 pixels (width * height). The value should be a data URI scheme 'data:image/<subtype>;base64,<data>'. (only if userId is specified)</td></tr>
<tr><td>userId</td><td>String</td><td></td><td></td><td>Openradiation.org user id</td></tr>
<tr><td>userPwd</td><td>String</td><td>Never</td><td></td><td>Openradiation.org plain text password (mandatory if userId is specified)</td></tr>
<tr><td>measurementEnvironment</td><td>String</td><td></td><td></td><td>Measurement environment : countryside, city, ontheroad, inside, plane (if plane, qualification is set to plane)</td></tr>
<tr><td>rain</td><td>Boolean</td><td></td><td></td><td>Rain : true if it rains during the measurement</td></tr>
<tr><td>flightNumber</td><td>String</td><td></td><td></td><td>if measurementEnvironment is plane, flightNumber of the commercial flight in capital letters (AITA code followed by number, example: AF179)</td></tr>
<tr><td>seatNumber</td><td>String</td><td></td><td></td><td>if measurementEnvironment is plane, seatNumber in capital letters with row number first (example: 14C)</td></tr>
<tr><td>windowSeat</td><td>Boolean</td><td></td><td></td><td>if measurementEnvironment is plane, windowSeat : true if the seat where is the sensor is next to the window</td></tr>
<tr><td>storm</td><td>Boolean</td><td></td><td></td><td>Storm : true if storm crossing during the measurement</td></tr>
<tr><td>flightId</td><td>Integer</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber setted, measurement is associated to a flightId (flightId permits to retrieve all measurements in a same flight)</td></tr>
<tr><td>refinedLatitude</td><td>Real</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, latitude is refined with the track of the plane</td></tr>
<tr><td>refinedLongitude</td><td>Real</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, longitude is refined with the track of the plane</td></tr>
<tr><td>refinedAltitude</td><td>Integer</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, altitude is refined with the track of the plane</td></tr>
<tr><td>refinedEndLatitude</td><td>Real</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, endLatitude is refined with the track of the plane</td></tr>
<tr><td>refinedEndLongitude</td><td>Real</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, endLongitude is refined with the track of the plane</td></tr>	
<tr><td>refinedEndAltitude</td><td>Integer</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, endAltitude is refined with the track of the plane</td></tr>
<tr><td>departureTime</td><td>Timestamp</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, actual departure time of the plane</td></tr>
<tr><td>arrivalTime</td><td>Timestamp</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, actual arrival time of the plane</td></tr>
<tr><td>airportOrigin</td><td>String</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, airport origin (AITA) of the plane</td></tr>
<tr><td>airportDestination</td><td>String</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, airport destination (AITA) of the plane</td></tr>	
<tr><td>aircraftType</td><td>String</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, aircraft type of the plane</td></tr>
<tr><td>firstLatitude</td><td>Real</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber setted, latitude of the first measurement or latitude of the origin airport</td></tr>
<tr><td>firstLongitude</td><td>Real</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber setted, longitude of the first measurement or longitude of the origin airport</td></tr>
<tr><td>midLatitude</td><td>Real</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber setted, latitude picked up from a measurement or in the middle of the plane track</td></tr>
<tr><td>midLongitude</td><td>Real</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber setted, longitude picked up from a measurement or in the middle of the plane track</td></tr>
<tr><td>lastLatitude</td><td>Real</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber setted, latitude of the last measurement or latitude of the destination airport</td></tr>
<tr><td>lastLongitude</td><td>Real</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber setted, longitude of the last measurement or longitude of the destination airport</td></tr>
<tr><td>dateAndTimeOfCreation</td><td>Timestamp</td><td></td><td>No, but always determinated by the API</td><td>Date of registration in the database</td></tr>
<tr><td>qualification</td><td>String</td><td>*</td><td>No, determinated by the API or the website</td><td>plane, wrongmeasurement, temporarysource, groundlevel</td></tr>
<tr><td>qualificationVotesNumber</td><td>Integer</td><td></td><td>No, determinated by the API or the website</td><td>qualification Votes Number</td></tr>
<tr><td>reliability</td><td>Integer</td><td></td><td>No, but always determinated by the API and never modified</td><td>Estimated reliability that the measurement is correct. Calculated when submitted to the API as following : +1 for each filled field, + min(30,HitsNumber) if HitsNumber not null, +10 if userId not null, +20 if ManualReporting=false, +20 if MeasurementEnvironment=countryside / +10 if MeasurementEnvironment=city or ontheroad, +10 if MeasurementHeight=1. Expecting > 78 (if not qualification is set to mustbeverified and qualificationVotesNumber is set to 0)</td></tr>
<tr><td>atypical</td><td>Boolean</td><td>*</td><td>No, but always determinated by the API and never modified</td><td>atypical if value is not representative of an environnemental measure. No if value < 0.2 (but we should compare value to an estimated local reference ...), yes otherwise</td></tr>
</table>


Observons la proportion des données manquantes sur chaque colonne.

In [35]:
analyser_dataframe(measurements)

StatementMeta(, 56081b58-39e9-42f3-b70c-38f20de855ab, 37, Finished, Available, Finished, False)

Nombre de lignes / colonnes 1992982 / 60

--- Structure des colonnes ---
root
 |-- apparatusId: string (nullable = true)
 |-- apparatusVersion: string (nullable = true)
 |-- apparatusSensorType: string (nullable = true)
 |-- apparatusTubeType: string (nullable = true)
 |-- temperature: string (nullable = true)
 |-- value: double (nullable = true)
 |-- hitsNumber: string (nullable = true)
 |-- calibrationFunction: string (nullable = true)
 |-- startTime: string (nullable = true)
 |-- endTime: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- accuracy: double (nullable = true)
 |-- altitude: integer (nullable = true)
 |-- altitudeAccuracy: double (nullable = true)
 |-- endLatitude: double (nullable = true)
 |-- endLongitude: double (nullable = true)
 |-- endAccuracy: string (nullable = true)
 |-- endAltitude: string (nullable = true)
 |-- endAltitudeAccuracy: string (nullable = true)
 |-- deviceUuid: string (nullable = true)
 |

Nous observons dans nos colonnes les plus importantes une présence de données manquantes :
- reportUuid : identifiant de la mesure (0.30%)
- startTime : date et temps du début de la mesure (0.30%)
- endTime : date et temps de la fin de la mesure (8.42%)
- dataAndTimeOfCreation : date et temps de la création de la mesure (0.74%)
- atypical : si la mesure est atypique (0.74%)

Le reste de ces colonnes concernent surtout des informations enregistrées en plus.

On y retrouve notamment les informations sur l'appareil, quelques informations météorologiques si renseignées, les vols concernées pour les mesures aux vols seulement et les coordonnées de la mesure du début à la fin ave ceux réajustées.

On observe donc beaucoup de colonnes dont la plupart ont une proportion de données manquantes élevées dû qu'il existe deux types de mesures (au sol et aérien).

## Observation de la structure pour measurements_withoutEnclosedObject.csv

In [36]:
analyser_dataframe(measurements_without_enclobject)

StatementMeta(, 56081b58-39e9-42f3-b70c-38f20de855ab, 38, Finished, Available, Finished, False)

Nombre de lignes / colonnes 1992984 / 59

--- Structure des colonnes ---
root
 |-- apparatusId: string (nullable = true)
 |-- apparatusVersion: string (nullable = true)
 |-- apparatusSensorType: string (nullable = true)
 |-- apparatusTubeType: string (nullable = true)
 |-- temperature: integer (nullable = true)
 |-- value: string (nullable = true)
 |-- hitsNumber: string (nullable = true)
 |-- calibrationFunction: string (nullable = true)
 |-- startTime: string (nullable = true)
 |-- endTime: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- accuracy: double (nullable = true)
 |-- altitude: double (nullable = true)
 |-- altitudeAccuracy: double (nullable = true)
 |-- endLatitude: double (nullable = true)
 |-- endLongitude: string (nullable = true)
 |-- endAccuracy: string (nullable = true)
 |-- endAltitude: string (nullable = true)
 |-- endAltitudeAccuracy: string (nullable = true)
 |-- deviceUuid: string (nullable = true)
 |

Le même jeu de données que `measurements.csv` mais avec une seule colonne en moins et quelques lignes de plus.

## Observation de la structure pour le fichier tags.csv

In [37]:
analyser_dataframe(tags)

StatementMeta(, 56081b58-39e9-42f3-b70c-38f20de855ab, 39, Finished, Available, Finished, False)

Nombre de lignes / colonnes 2337807 / 2

--- Structure des colonnes ---
root
 |-- reportUuid: string (nullable = true)
 |-- tag: string (nullable = true)


--- Données manquantes par colonne ---
Colonne : reportUuid | Type : string | Manquants : 0 | Proportion : 0.00 %
Colonne : tag | Type : string | Manquants : 0 | Proportion : 0.00 %


In [38]:
tags.show()

StatementMeta(, 56081b58-39e9-42f3-b70c-38f20de855ab, 41, Finished, Available, Finished, False)

+--------------------+--------------------+
|          reportUuid|                 tag|
+--------------------+--------------------+
|00000000-0000-000...|              icohup|
|00000000-0000-000...|serie_46be4e4ce77...|
|0000046a-3759-4da...|series_0c81ce7984...|
|00000b30-1748-4ab...|series_e3ae9959d8...|
|00003f37-7db6-453...|series_28a15ac1f5...|
|00004030-ddc2-488...|series_f7a95040a6...|
|000056cd-6f32-420...|series_bd86100b02...|
|00007ff4-1331-495...|series_99bae0ec4b...|
|0000e6ad-af08-4d9...| arrivée: barcelone |
|0000e6ad-af08-4d9...|       départ: paris|
|0000e6ad-af08-4d9...|serie_35b93e05f3f...|
|00010b6a-c695-474...|        arrivée: cdg|
|00010b6a-c695-474...|         départ: yvr|
|00010b6a-c695-474...|serie_6580d7eccf9...|
|00010f85-b977-4e0...|series_e390d3e484...|
|00011286-55ea-4a6...|series_65ac709369...|
|000113b6-2cc3-4ad...|     challenge sirse|
|000113b6-2cc3-4ad...|series_9ebcb4fa63...|
|000147e2-66df-4e3...|series_109a8dd12c...|
|00019069-54aa-4d1...|series_93b

In [41]:
tags.select("reportUuid").head(10)

StatementMeta(, 56081b58-39e9-42f3-b70c-38f20de855ab, 45, Finished, Available, Finished, False)

[Row(reportUuid='00000000-0000-0000-0000-000000000000'),
 Row(reportUuid='00000000-0000-0000-0000-000000000000'),
 Row(reportUuid='0000046a-3759-4dae-9ff8-4d138c5bc858'),
 Row(reportUuid='00000b30-1748-4ab7-8212-6b77ad67dabe'),
 Row(reportUuid='00003f37-7db6-453d-9fca-3cbafe417978'),
 Row(reportUuid='00004030-ddc2-4880-b7f6-2fa04c5678d0'),
 Row(reportUuid='000056cd-6f32-4202-83ad-f03f997ee7d7'),
 Row(reportUuid='00007ff4-1331-495b-9fc0-8f1c04e38332'),
 Row(reportUuid='0000e6ad-af08-4d98-9799-85910bb0fab1'),
 Row(reportUuid='0000e6ad-af08-4d98-9799-85910bb0fab1')]

Ici, on voit que les tags sont associées à une mesure via l'identifiant reportUuid.

In [42]:
tags.filter("reportUuid == '0000e6ad-af08-4d98-9799-85910bb0fab1'").show()

StatementMeta(, 56081b58-39e9-42f3-b70c-38f20de855ab, 46, Finished, Available, Finished, False)

+--------------------+--------------------+
|          reportUuid|                 tag|
+--------------------+--------------------+
|0000e6ad-af08-4d9...| arrivée: barcelone |
|0000e6ad-af08-4d9...|       départ: paris|
|0000e6ad-af08-4d9...|serie_35b93e05f3f...|
+--------------------+--------------------+



La relation entre tags et mesures est selon le type de mesure. 

Pour une mesure au sol, il n'y aura qu'un seul tag (le nom de la mesure)
tandis que pour une mesure au vol, il faudra compter trois tags (tag de départ, d'arrivé et de la série). 

Il y faudra faire attention à la jointure entre les deux jeux de données.